# Aula 2 · Treinamento, avaliação e mudança de distribuição

Notebook da Aula 2 do guia **Visão computacional aplicada com CVAT e YOLO** (github.com/tassid/pdi-agro).

Treina-se um detector YOLO no WGISD (Embrapa) com **treino** em Chardonnay, Cabernet Franc e Cabernet Sauvignon, **validação** em Sauvignon Blanc e **teste** em Syrah, variedade ausente do treinamento.

Ambiente de execução → **GPU (T4)** antes de rodar a primeira célula.

## 1. Ambiente

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics
import sys, torch, ultralytics
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)

## 2. Dados com partição por variedade (definida na Aula 1)

In [ ]:
# Baixa o WGISD (Embrapa): só as anotações e as fotos, sem histórico do git
import os, subprocess
if not os.path.exists("wgisd"):
    subprocess.run("git clone --depth 1 https://github.com/thsant/wgisd.git", shell=True, check=True)
import glob, random, shutil, yaml
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
fotos = sorted(glob.glob("wgisd/data/*.jpg"))
df = pd.DataFrame({"foto": fotos})
df["nome"] = df.foto.str.split("/").str[-1].str.replace(".jpg", "", regex=False)
df["variedade"] = df.nome.str[:3]
df["caixas"] = [sum(1 for l in open(f.replace(".jpg", ".txt")) if l.strip()) for f in df.foto]
df.groupby("variedade").agg(fotos=("nome", "size"), caixas=("caixas", "sum"))

PAPEL = {"CDY": "train", "CFR": "train", "CSV": "train", "SVB": "val", "SYH": "test"}
df["conjunto"] = df.variedade.map(PAPEL)
SAIDA = "wgisd_yolo"
shutil.rmtree(SAIDA, ignore_errors=True)
for _, r in df.iterrows():
    for sub in ["images", "labels"]:
        os.makedirs(f"{SAIDA}/{sub}/{r.conjunto}", exist_ok=True)
    im = Image.open(r.foto).convert("RGB"); im.thumbnail((1280, 1280))
    im.save(f"{SAIDA}/images/{r.conjunto}/{r.nome}.jpg", quality=92)
    shutil.copy(r.foto.replace(".jpg", ".txt"), f"{SAIDA}/labels/{r.conjunto}/{r.nome}.txt")
DATA = os.path.abspath(f"{SAIDA}/data.yaml")
yaml.safe_dump({"path": os.path.abspath(SAIDA), "train": "images/train", "val": "images/val",
                "test": "images/test", "names": {0: "grape"}}, open(DATA, "w"))
print(df.groupby("conjunto").agg(fotos=("nome", "size"), caixas=("caixas", "sum")))

## 2b. Perceptron e perceptron multicamadas

In [ ]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
for nome, y in {"AND": [0,0,0,1], "OR": [0,1,1,1], "XOR": [0,1,1,0]}.items():
    w, b, eta = np.zeros(2), 0.0, 0.2
    for ep in range(30):
        erros = 0
        for xi, yi in zip(X, y):
            e = yi - int(w @ xi + b > 0); w += eta * e * xi; b += eta * e; erros += abs(e)
        if erros == 0: break
    print(f"{nome}: w={w}, b={b:.1f}, {'convergiu na época ' + str(ep + 1) if erros == 0 else 'não converge'}")

# MLP 2-4-1 resolve o XOR
yx = np.array([[0],[1],[1],[0]], float); r = np.random.default_rng(1)
W1, b1, W2, b2 = r.normal(size=(2,4)), np.zeros(4), r.normal(size=(4,1)), np.zeros(1)
sig = lambda z: 1 / (1 + np.exp(-z))
for it in range(4000):
    h = np.tanh(X @ W1 + b1); o = sig(h @ W2 + b2)
    do = (o - yx) * o * (1 - o); dh = do @ W2.T * (1 - h**2)
    W2 -= h.T @ do; b2 -= do.sum(0); W1 -= X.T @ dh; b1 -= dh.sum(0)
print("MLP no XOR:", o.round(2).ravel())

## 3. Treinamento

Transferência de aprendizado a partir de pesos treinados no COCO, com parada antecipada. Verifique na documentação da versão instalada os pesos disponíveis (`yolo11n.pt`, `yolo26n.pt`, entre outros).

In [ ]:
from ultralytics import YOLO
PESOS = "yolo11n.pt"
modelo = YOLO(PESOS)
modelo.train(data=DATA, epochs=40, imgsz=640, batch=16, patience=15, seed=42,
             project="runs", name="uva_n", exist_ok=True, plots=True)
PASTA = "runs/uva_n"

## 4. Curvas de aprendizado e métricas no conjunto de validação

In [ ]:
res = pd.read_csv(f"{PASTA}/results.csv"); res.columns = res.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for c in ["train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss"]:
    if c in res: ax[0].plot(res.epoch, res[c], label=c)
ax[0].set(xlabel="época", title="Perdas"); ax[0].legend()
ax[1].plot(res.epoch, 100 * res["metrics/mAP50(B)"], label="mAP50")
ax[1].plot(res.epoch, 100 * res["metrics/mAP50-95(B)"], label="mAP50-95")
ax[1].set(xlabel="época", ylabel="%", title="Validação"); ax[1].legend(); plt.show()
print("melhor época:", int(res.epoch[res["metrics/mAP50(B)"].idxmax()]), "| última:", int(res.epoch.iloc[-1]))

In [ ]:
melhor = YOLO(f"{PASTA}/weights/best.pt")
val = melhor.val(data=DATA, split="val", plots=True, verbose=False)
print(f"VALIDAÇÃO  P={val.box.mp:.3f}  R={val.box.mr:.3f}  mAP50={val.box.map50:.3f}  mAP50-95={val.box.map:.3f}")

### Limiar de confiança e erro de contagem

In [ ]:
fotos_val = sorted(glob.glob(f"{SAIDA}/images/val/*.jpg"))
linhas = []
for conf in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    erros = []
    for f in fotos_val:
        n_real = sum(1 for l in open(f.replace("images", "labels").replace(".jpg", ".txt")) if l.strip())
        n_prev = len(melhor.predict(f, conf=conf, imgsz=640, verbose=False)[0].boxes)
        erros.append(n_prev - n_real)
    linhas.append({"conf": conf, "erro médio": np.mean(erros), "erro absoluto médio": np.mean(np.abs(erros))})
pd.DataFrame(linhas)

## 5. Avaliação fora da distribuição: variedade Syrah, ausente do treino

In [ ]:
teste = melhor.val(data=DATA, split="test", plots=True, verbose=False)
pd.DataFrame({"validação (Sauvignon Blanc)": [val.box.mp, val.box.mr, val.box.map50, val.box.map],
              "teste (Syrah)": [teste.box.mp, teste.box.mr, teste.box.map50, teste.box.map]},
             index=["P", "R", "mAP50", "mAP50-95"]).round(3)

In [ ]:
# 5 fotos do teste: anotação (esquerda) contra predição (direita). Classifique cada erro.
for f in sorted(glob.glob(f"{SAIDA}/images/test/*.jpg"))[:5]:
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    im = Image.open(f); W, H = im.size; ax[0].imshow(im); ax[0].set_title("anotação"); ax[0].axis("off")
    for l in open(f.replace("images", "labels").replace(".jpg", ".txt")):
        _, xc, yc, w, h = map(float, l.split())
        ax[0].add_patch(patches.Rectangle(((xc - w/2) * W, (yc - h/2) * H), w * W, h * H, fill=False, ec="lime", lw=2))
    ax[1].imshow(melhor.predict(f, conf=0.3, verbose=False)[0].plot()[:, :, ::-1]); ax[1].set_title("predição"); ax[1].axis("off")
    plt.show()

### 5b. Teste de robustez a perturbações sintéticas

As imagens de validação são submetidas a perturbações de iluminação, cor, ruído e nitidez. Identifique a perturbação de maior efeito sobre a razão entre caixas previstas e anotadas. Perturbações sintéticas não substituem a avaliação com dados de outro domínio real.

In [ ]:
import cv2
def transforma(im, tipo):
    f = im.astype(np.float32) / 255
    if tipo == "fim de tarde": f = f ** 1.8
    if tipo == "contraluz": f = f ** 0.5
    if tipo == "balanço quente": f = np.clip(f * [0.75, 1.0, 1.15], 0, 1)   # BGR
    if tipo == "ruído": f = np.clip(f + np.random.default_rng(0).normal(0, 0.06, f.shape), 0, 1)
    if tipo == "desfoque": f = cv2.GaussianBlur(f, (0, 0), 3)
    return (f * 255).astype(np.uint8)

linhas = []
for tipo in ["original", "fim de tarde", "contraluz", "balanço quente", "ruído", "desfoque"]:
    acertos = reais = previstas = 0
    for f in fotos_val:
        im = transforma(cv2.imread(f), tipo)
        n_real = sum(1 for l in open(f.replace("images", "labels").replace(".jpg", ".txt")) if l.strip())
        n_prev = len(melhor.predict(im, conf=0.3, imgsz=640, verbose=False)[0].boxes)
        reais += n_real; previstas += n_prev
    linhas.append({"condição": tipo, "cachos reais": reais, "caixas previstas": previstas, "razão": round(previstas / reais, 2)})
pd.DataFrame(linhas)

## 6. Rastreamento e contagem em vídeo

Na ausência de um vídeo do vinhedo, é gerada uma sequência sintética deslocando uma janela ao longo de uma imagem de Syrah. Substitua `VIDEO` por um vídeo real quando disponível.

In [ ]:
import cv2
foto = sorted(glob.glob(f"{SAIDA}/images/test/*.jpg"))[0]
im = cv2.imread(foto); H, W = im.shape[:2]; jan = int(W * 0.5)
VIDEO = "fileira.mp4"
out = cv2.VideoWriter(VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), 15, (jan, H))
for x0 in range(0, W - jan, 8):
    out.write(im[:, x0:x0 + jan])
out.release()

ids, caixas = set(), 0
for r in melhor.track(VIDEO, tracker="bytetrack.yaml", conf=0.3, stream=True, verbose=False, persist=True):
    caixas += len(r.boxes)
    if r.boxes.id is not None: ids.update(r.boxes.id.int().tolist())
n_real = sum(1 for l in open(foto.replace("images", "labels").replace(".jpg", ".txt")) if l.strip())
print(f"soma de caixas em todos os quadros: {caixas} | ids únicos: {len(ids)} | cachos anotados na foto: {n_real}")

## 7. Busca aleatória de hiperparâmetros (opcional, de maior duração)

A seleção utiliza exclusivamente o conjunto de validação (BERGSTRA; BENGIO, 2012).

In [ ]:
RODAR_BUSCA = False
if RODAR_BUSCA:
    rng = np.random.default_rng(0); busca = []
    for k in range(4):
        lr0 = float(10 ** rng.uniform(-3.5, -1.5)); imgsz = int(rng.choice([512, 640, 800]))
        YOLO(PESOS).train(data=DATA, epochs=20, imgsz=imgsz, lr0=lr0, batch=16, patience=8, seed=k,
                          project="runs", name=f"busca{k}", exist_ok=True, plots=False, verbose=False)
        m = YOLO(f"runs/busca{k}/weights/best.pt").val(data=DATA, split="val", verbose=False)
        busca.append({"lr0": lr0, "imgsz": imgsz, "mAP50 val": m.box.map50})
    display(pd.DataFrame(busca).sort_values("mAP50 val", ascending=False))

## 8. Exportação e registro dos elementos necessários à reprodução

In [ ]:
melhor.export(format="onnx", imgsz=640)
import json, platform
json.dump({"pesos_iniciais": PESOS, "divisao": PAPEL, "ultralytics": ultralytics.__version__, "torch": torch.__version__,
           "python": platform.python_version(), "val": {"mAP50": val.box.map50, "mAP50-95": val.box.map},
           "teste": {"mAP50": teste.box.map50, "mAP50-95": teste.box.map}},
          open(f"{PASTA}/reprodutibilidade.json", "w"), indent=2)
!ls {PASTA} {PASTA}/weights
# from google.colab import drive; drive.mount("/content/drive"); !cp -r runs /content/drive/MyDrive/

## Exercícios

1. Repita o treinamento com `imgsz=960` e `batch=8`. A variação do mAP50-95 é maior que a do mAP50? Justifique.
2. Inverta os papéis das variedades: treine em Syrah, Cabernet Sauvignon e Cabernet Franc e avalie em Chardonnay. A diferença entre validação e teste é simétrica?
3. Treine com `freeze=10` e compare com o treinamento completo no conjunto de validação.
4. Execute `model.tune(data=DATA, epochs=10, iterations=8, plots=False, val=True)` e compare a configuração encontrada com a padrão no conjunto de validação.
5. Selecione `conf` pelo menor erro absoluto de contagem na validação e, apenas então, meça o erro de contagem no teste.

## Referências

BERGSTRA, J.; BENGIO, Y. Random search for hyper-parameter optimization. Journal of Machine Learning Research, v. 13, p. 281-305, 2012.

KOH, P. W. et al. WILDS: a benchmark of in-the-wild distribution shifts. In: ICML, 38., 2021. Proceedings [...]. PMLR, v. 139, p. 5637-5664, 2021.

SANTOS, T. T. et al. Grape detection, segmentation, and tracking using deep neural networks and three-dimensional association. Computers and Electronics in Agriculture, v. 170, 105247, 2020.

ZHANG, Y. et al. ByteTrack: multi-object tracking by associating every detection box. In: ECCV, 2022. Proceedings [...]. Cham: Springer, 2022. p. 1-21.